# MPPI Control Policy with Less Strict CBF

### Import necessary packages

In [8]:
import math
import numpy as np
import jax
import jax.numpy as jnp
import dynamaxsys
from dynamaxsys.simplecar import DynamicallyExtendedSimpleCar

### Define robot and human dynamics

In [9]:
# --- Dynamics setup ---
WHEELBASE = 0.5   # m
DT        = 0.1   # s

# Continuous-time dynamics, then wrapped with RK4 integrator
_continuous = DynamicallyExtendedSimpleCar(wheelbase=WHEELBASE)
_discrete   = dynamaxsys.get_discrete_time_dynamics(_continuous, DT)

def step_fn(state: jnp.ndarray, control: jnp.ndarray) -> jnp.ndarray:
    """Single step: state [px, py, theta, v] -> next state."""
    return _discrete(state, control)

# Vectorized version: takes (N, 4) states and (N, 2) controls -> (N, 4) next states
step_batch = jax.vmap(step_fn, in_axes=(0, 0))

### Define MPPI rollout function

In [10]:
# --- MPPI parameters (just what's needed for the rollout) ---
HORIZON    = 20     # planning horizon (steps)
N_SAMPLES  = 500    # number of perturbed trajectories

# Control limits (same as CBF notebook)
A_MIN,         A_MAX         = -5.0, 5.0
TAN_DELTA_MIN, TAN_DELTA_MAX = -0.8, 0.8

def rollout_single(state0: jnp.ndarray, controls: jnp.ndarray) -> jnp.ndarray:
    """
    Roll out dynamics from state0 for one sample.

    Args:
        state0:   (4,)           initial state [px, py, theta, v]
        controls: (HORIZON, 2)   control sequence for this sample

    Returns:
        states:   (HORIZON+1, 4) trajectory including the initial state
    """
    def scan_fn(state, control):
        next_state = step_fn(state, control)
        return next_state, next_state   # (carry, output)

    _, states = jax.lax.scan(scan_fn, state0, controls)
    # states is (HORIZON, 4) — prepend state0 to get full trajectory
    states = jnp.concatenate([state0[None, :], states], axis=0)
    return states   # (HORIZON+1, 4)


# Vectorize over the sample dimension:
# each sample gets the same state0 but a different control sequence
rollout_batch = jax.vmap(rollout_single, in_axes=(None, 0))

# JIT compile for speed — this makes subsequent calls much faster
rollout_batch = jax.jit(rollout_batch)

### Define the cost function used to get the best MPPI trajectories

In [11]:
# --- Cost function parameters ---
# Goal
Q_GOAL       = 10.0   # weight on distance to goal at each step
Q_GOAL_FINAL = 50.0   # extra weight on final state (terminal cost)

# Human proximity
D_STOP        = 3.0    # hard barrier radius — infinite cost inside this
D_SOFT        = 10.0   # soft penalty starts here
Q_HUMAN_SOFT  = 5.0    # weight on soft proximity penalty
Q_HUMAN_HARD  = 1e6    # large finite number standing in for "infinite" cost

# Control effort (optional regularization)
Q_CONTROL     = 0.01   # small penalty on large controls

# Goal position (will be set before running)
GOAL_X = 50.0
GOAL_Y = 20.0


def human_cost_single(pos: jnp.ndarray, human_positions: jnp.ndarray) -> jnp.ndarray:
    """
    Cost at a single robot position due to all humans.

    Args:
        pos:             (2,)         robot [px, py]
        human_positions: (N_HUMANS, 2)

    Returns:
        scalar cost
    """
    # distances from this position to every human: (N_HUMANS,)
    diffs = pos[None, :] - human_positions          # (N_HUMANS, 2)
    dists = jnp.linalg.norm(diffs, axis=1)          # (N_HUMANS,)

    # soft penalty: grows as robot enters D_SOFT radius
    # uses 1/d shape so cost rises steeply close in
    soft = jnp.sum(
        jnp.where(dists < D_SOFT,
                  Q_HUMAN_SOFT * (D_SOFT - dists) / D_SOFT,
                  0.0)
    )

    # hard barrier: enormous cost if inside D_STOP
    hard = jnp.sum(
        jnp.where(dists < D_STOP,
                  Q_HUMAN_HARD,
                  0.0)
    )

    return soft + hard


def trajectory_cost(
    trajectory:      jnp.ndarray,   # (HORIZON+1, 4)
    controls:        jnp.ndarray,   # (HORIZON, 2)
    human_positions: jnp.ndarray,   # (N_HUMANS, 2)
) -> jnp.ndarray:
    """
    Total cost for a single sample trajectory.

    Sums three terms over the horizon:
      1. Distance to goal at each step
      2. Human proximity penalty at each step
      3. Control effort
    Plus a terminal goal cost on the final state.
    """
    goal = jnp.array([GOAL_X, GOAL_Y])

    # --- per-step costs (over steps 0..HORIZON-1) ---
    states = trajectory[:-1]                              # (HORIZON, 4)
    positions = states[:, :2]                             # (HORIZON, 2)

    # 1. goal distance at each step
    goal_dists  = jnp.linalg.norm(positions - goal, axis=1)    # (HORIZON,)
    goal_cost   = Q_GOAL * jnp.sum(goal_dists)

    # 2. human proximity at each step — vmap over timesteps
    human_cost_per_step = jax.vmap(
        human_cost_single, in_axes=(0, None)
    )(positions, human_positions)                               # (HORIZON,)
    human_cost  = jnp.sum(human_cost_per_step)

    # 3. control effort
    control_cost = Q_CONTROL * jnp.sum(controls ** 2)

    # --- terminal cost: heavily penalize distance to goal at final state ---
    final_pos    = trajectory[-1, :2]                          # (2,)
    terminal_cost = Q_GOAL_FINAL * jnp.linalg.norm(final_pos - goal)

    return goal_cost + human_cost + control_cost + terminal_cost


# Vectorize over samples: (N_SAMPLES, HORIZON+1, 4) -> (N_SAMPLES,)
cost_batch = jax.vmap(trajectory_cost, in_axes=(0, 0, None))
cost_batch = jax.jit(cost_batch)

In [12]:
# --- Sanity check ---
import jax.random as jr

key = jr.PRNGKey(0)
human_positions = jnp.array([
    [20.0, 20.0],
    [35.0, 18.0],
    [45.0, 22.0],
    [30.0, 25.0],
])

# random controls for all samples
E = jr.normal(key, shape=(N_SAMPLES, HORIZON, 2)) * 0.5
U_nom = jnp.zeros((HORIZON, 2))
perturbed = U_nom[None] + E

state0 = jnp.array([5.0, 20.0, 0.0, 0.0])
trajs  = rollout_batch(state0, perturbed)
costs  = cost_batch(trajs, perturbed, human_positions)

print("Costs shape:", costs.shape)          # (500,)
print("Min cost:  ", costs.min())
print("Max cost:  ", costs.max())
print("Mean cost: ", costs.mean())

Costs shape: (500,)
Min cost:   11146.893
Max cost:   11334.803
Mean cost:  11250.781
